# Fashion-MNIST Image Compression with Autoencoders

**Deep Learning (2087101)**

This notebook implements an undercomplete autoencoder and compares latent dimensions 8, 32, and 64. It also implements a denoising autoencoder. Run all cells from top to bottom to generate the actual MSE values, compression ratios, loss curves, and image visualizations.

**Dataset:** Fashion-MNIST (28 × 28 grayscale images)  
**Framework:** TensorFlow / Keras  
**Metrics:** Mean Squared Error (MSE), theoretical compression ratio

In [ ]:
# Install TensorFlow if needed (uncomment in Google Colab if it is missing)
# %pip install tensorflow matplotlib numpy

import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model

print("TensorFlow version:", tf.__version__)
np.random.seed(42)
tf.random.set_seed(42)

## 1. Load and prepare Fashion-MNIST

Pixel values are scaled from 0–255 to 0–1. Each 28 × 28 image is flattened into a vector of 784 values for the fully connected autoencoder.

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print("Training samples:", x_train.shape[0])
print("Testing samples:", x_test.shape[0])
print("Input dimensions:", x_train.shape[1])

## 2. Build an undercomplete autoencoder

The encoder compresses 784 input values into a smaller latent vector. The decoder reconstructs the image from that vector. The latent dimension is kept much smaller than 784.

In [ ]:
def build_autoencoder(latent_dim):
    inputs = layers.Input(shape=(784,), name="image_input")
    encoded = layers.Dense(128, activation="relu", name="encoder_hidden")(inputs)
    latent = layers.Dense(latent_dim, activation="relu", name="latent_vector")(encoded)
    decoded = layers.Dense(128, activation="relu", name="decoder_hidden")(latent)
    outputs = layers.Dense(784, activation="sigmoid", name="reconstructed_image")(decoded)

    autoencoder = Model(inputs, outputs, name=f"autoencoder_{latent_dim}")
    encoder = Model(inputs, latent, name=f"encoder_{latent_dim}")
    autoencoder.compile(optimizer="adam", loss="mse")
    return autoencoder, encoder

def calculate_mse(model, test_data):
    reconstructed = model.predict(test_data, verbose=0)
    mse = np.mean((test_data - reconstructed) ** 2)
    return mse, reconstructed

latent_sizes = [8, 32, 64]
models = {}
encoders = {}
results = {}

for size in latent_sizes:
    print(f"\nTraining latent size {size}...")
    autoencoder, encoder = build_autoencoder(size)
    start = time.time()
    history = autoencoder.fit(
        x_train, x_train,
        epochs=10,
        batch_size=256,
        shuffle=True,
        validation_split=0.1,
        verbose=1
    )
    elapsed = time.time() - start
    mse, reconstructed = calculate_mse(autoencoder, x_test)

    # Each original image has 784 values; the compressed representation has 'size' values.
    compression_ratio = 784 / size
    models[size] = autoencoder
    encoders[size] = encoder
    results[size] = {
        "mse": float(mse),
        "compression_ratio": float(compression_ratio),
        "history": history.history,
        "reconstructed": reconstructed,
        "training_seconds": elapsed
    }
    print(f"Latent size: {size}")
    print(f"Test MSE: {mse:.6f}")
    print(f"Approx. compression ratio (input values / latent values): {compression_ratio:.2f}:1")
    print(f"Training time: {elapsed:.1f} seconds")

## 3. Compare bottleneck dimensions

The ratio below compares the number of input pixel values with the number of latent values. It is a **theoretical value-count ratio**, not the exact disk-space ratio of a saved model. The latent vectors are floating-point values, and model weights are not included.

In [ ]:
print(f"{'Latent size':<15}{'Test MSE':<15}{'Value-count ratio':<20}")
for size in latent_sizes:
    print(f"{size:<15}{results[size]['mse']:<15.6f}{results[size]['compression_ratio']:.2f}:1")

plt.figure(figsize=(7, 4))
for size in latent_sizes:
    plt.plot(results[size]["history"]["loss"], label=f"Train, latent={size}")
plt.title("Training loss by latent dimension")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
for size in latent_sizes:
    plt.plot(results[size]["history"]["val_loss"], label=f"Validation, latent={size}")
plt.title("Validation loss by latent dimension")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Display original images and their reconstructions for each latent size.
count = 8
fig, axes = plt.subplots(1 + len(latent_sizes), count, figsize=(14, 6))

for i in range(count):
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap="gray")
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("Original")

for row, size in enumerate(latent_sizes, start=1):
    for i in range(count):
        axes[row, i].imshow(results[size]["reconstructed"][i].reshape(28, 28), cmap="gray")
        axes[row, i].axis("off")
        if i == 0:
            axes[row, i].set_ylabel(f"Latent {size}", rotation=0, labelpad=35, va="center")

plt.tight_layout()
plt.show()

## 4. Denoising autoencoder

The model receives noisy images as input but uses clean images as the target. This teaches it to remove some random noise during reconstruction.

In [ ]:
def add_noise(images, noise_factor=0.3):
    noisy = images + noise_factor * np.random.normal(
        loc=0.0, scale=1.0, size=images.shape
    ).astype("float32")
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)

denoiser, denoising_encoder = build_autoencoder(32)
denoising_history = denoiser.fit(
    x_train_noisy, x_train,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_split=0.1,
    verbose=1
)

denoised = denoiser.predict(x_test_noisy, verbose=0)
denoising_mse = np.mean((x_test - denoised) ** 2)
print(f"Denoising autoencoder test MSE against clean images: {denoising_mse:.6f}")

In [ ]:
# Compare clean, noisy, and reconstructed images.
count = 8
fig, axes = plt.subplots(3, count, figsize=(14, 5))

for i in range(count):
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap="gray")
    axes[1, i].imshow(x_test_noisy[i].reshape(28, 28), cmap="gray")
    axes[2, i].imshow(denoised[i].reshape(28, 28), cmap="gray")
    for row in range(3):
        axes[row, i].axis("off")

axes[0, 0].set_ylabel("Original", rotation=0, labelpad=45, va="center")
axes[1, 0].set_ylabel("Noisy", rotation=0, labelpad=45, va="center")
axes[2, 0].set_ylabel("Denoised", rotation=0, labelpad=45, va="center")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(denoising_history.history["loss"], label="Training loss")
plt.plot(denoising_history.history["val_loss"], label="Validation loss")
plt.title("Denoising autoencoder loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True)
plt.show()

## 5. Results and discussion

Use the printed test MSE values from your run in the report. Usually, increasing the latent size gives the model more capacity to preserve image details, so reconstruction error often decreases. However, this is an empirical trend, not a guarantee; results depend on training settings and random initialization.

- **Latent size 8:** strongest compression, but may lose more fine details.
- **Latent size 32:** moderate compression and representation capacity.
- **Latent size 64:** lowest compression among these choices, with more capacity to preserve details.
- **Denoising model:** learns to map noisy inputs to clean target images.

The value-count ratios are 98:1, 24.5:1, and 12.25:1 for latent sizes 8, 32, and 64 respectively (784 divided by the latent size). These ratios exclude model parameters and storage overhead.

## Conclusion

An undercomplete autoencoder compresses Fashion-MNIST images into a smaller latent representation and reconstructs them with some information loss. Comparing latent sizes shows the trade-off between compression and reconstruction quality. The denoising autoencoder is trained with noisy inputs and clean targets, allowing it to learn noise-resistant representations. The final MSE values and plots should be taken from the actual notebook execution.